# Anexo A. Reproducibilidad

Este anexo reúne lo necesario para repetir el trabajo de extremo a extremo: el entorno con versiones fijadas, el hardware, la configuración de Spark, las semillas, el manifiesto de los datos y los comandos de ejecución.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from src import config as C
from src import plots as P
from src import report as R
from src.persist import load_json

P.setup()
C.ensure_dirs()
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 90)
print("Resultados:", C.RESULTS.relative_to(C.ROOT), "| prueba de humo:", C.SMOKE)

import importlib.metadata as md
import platform
import subprocess
from src import data as D

Resultados: results | prueba de humo: False


## A.1 Entorno de software

El entorno se crea con `python -m venv .venv` e instala las versiones fijadas de `requirements.txt` (alternativa con conda: `environment.yml`). PySpark necesita además Java 17 o posterior y, en Windows, los binarios de Hadoop `winutils.exe` y `hadoop.dll` en `C:\hadoop\bin`. La memoria de la JVM de Spark se puede ajustar con la variable de entorno `AVAZU_SPARK_MEMORY`.

In [2]:
paquetes = ["numpy", "pandas", "polars", "duckdb", "pyarrow", "scipy", "scikit-learn", "statsmodels", "optuna", "pyspark",
            "lime", "shap", "matplotlib", "psutil", "jupyter-book"]
ver = {}
for p in paquetes:
    try:
        ver[p] = md.version(p)
    except md.PackageNotFoundError:
        ver[p] = "no instalado"
try:
    java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()
    java = next(l for l in java if "version" in l)
except Exception as e:  # noqa: BLE001
    java = f"no disponible ({type(e).__name__})"
ver = {"Python": platform.python_version(), **ver, "Java": java}
display(pd.Series(ver, name="versión").to_frame())

,versión
Python,3.12.10
numpy,2.5.4
pandas,2.3.3
polars,2.1.0
duckdb,2.0.0
pyarrow,26.0.1
scipy,1.18.1
scikit-learn,1.9.1
statsmodels,no instalado
optuna,5.0.0


## A.2 Hardware y configuración de Spark

In [3]:
hw = load_json(C.COMMON / "hardware.json")
display(pd.Series(hw["hardware"], name="valor").to_frame())
if (C.COMMON / "hardware_benchmark.json").exists():
    display(Markdown("**Segundo equipo.** PySpark, el escalado de scikit-learn, el escalamiento y el benchmark de lectura se ejecutaron en este equipo:"))
    display(pd.Series(load_json(C.COMMON / "hardware_benchmark.json")["hardware"], name="valor").to_frame())
info = load_json(C.COMMON / "spark_info.json")
conf = {k: v for k, v in info["conf"].items() if k in ("spark.master", "spark.driver.memory", "spark.driver.maxResultSize",
        "spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled", "spark.sql.execution.arrow.pyspark.enabled")}
conf.update({"versión": info["version"], "paralelismo por defecto": info["default_parallelism"],
             "núcleos (escalamiento)": ", ".join(map(str, C.SPARK_CORES_SCALING)), "control de memoria del MLP": f"memoria libre de la JVM ≥ {C.SPARK_MLP_MEM_FACTOR:g} × entrada apilada"})
display(pd.Series(conf, name="valor").to_frame())

,valor
sistema,Windows 11
procesador,"AMD64 Family 25 Model 117 Stepping 2, AuthenticAMD"
nucleos_fisicos,8
nucleos_logicos,16
frecuencia_max_mhz,3801.0
ram_gb,31.3
python,3.12.10


,valor
spark.driver.memory,16g
spark.driver.maxResultSize,4g
spark.sql.shuffle.partitions,32
spark.sql.execution.arrow.pyspark.enabled,true
spark.sql.adaptive.enabled,true
spark.master,local[8]
versión,4.3.0
paralelismo por defecto,8
núcleos (escalamiento),"1, 2, 4, 8"
control de memoria del MLP,memoria libre de la JVM ≥ 1 × entrada apilada


## A.3 Semillas y decisiones fijas

| Elemento | Valor |
|---|---|
| Semillas de entrenamiento | 0, 1, 2, 3, 4 (3 en los tamaños de 3 y 10 millones de la curva de escalado) |
| Semilla de Optuna (TPE) | 100 |
| Semilla del bootstrap y de la selección de casos de LIME | 1234 |
| Semillas de LIME | 0 a 9 |
| Semilla de la función hash del preprocesamiento (Polars) | 42 |
| Clave de muestreo | $u = \text{hash}(\text{id}) / 2^{64}$, calculada por DuckDB al convertir a Parquet |

Todas las decisiones del experimento están en `src/config.py`.

## A.4 Manifiesto de los datos

El repositorio no contiene los datos. `data/SHA256SUMS.txt` guarda la huella del archivo original y de cada archivo Parquet, y `data/manifest.json` las filas por día y la huella del conjunto de prueba. Al ejecutar `python -m src.runner prepare` con el mismo `train.gz`, las huellas deben coincidir.

In [4]:
man = D.load_manifest()
fpd = pd.Series({int(k): v for k, v in man["filas_por_dia"].items()}, name="filas")
arch = pd.DataFrame(man["archivos_parquet"])
arch["MB"] = (arch["bytes"] / 1024**2).round(1)
arch["sha256"] = arch["sha256"].str[:16] + "…"
display(arch[["archivo", "MB", "sha256"]].set_index("archivo"))
print("Archivo original:", man["raw"]["archivo"], f"({man['raw']['bytes'] / 1024**3:.2f} GB)")
print("SHA-256 original:", load_json(C.COMMON / "raw_sha256.json")["sha256"])
print("Huella del conjunto de prueba:", man["test_sha256"])
D.verify_test_frozen(); print("Verificación del conjunto de prueba: correcta")

,MB,sha256
archivo,,
parquet/avazu/day=21/data_0.parquet,93.2,9aa0d27efdc2d833…
parquet/avazu/day=22/data_0.parquet,117.3,436936d6140876b0…
parquet/avazu/day=23/data_0.parquet,87.1,e9a4c08702023c24…
parquet/avazu/day=24/data_0.parquet,71.9,137894e506ef9fea…
parquet/avazu/day=25/data_0.parquet,74.1,8d4a8191066ada69…
parquet/avazu/day=26/data_0.parquet,84.4,06de86b379c35625…
parquet/avazu/day=27/data_0.parquet,73.1,4353e45265ba5e8a…
parquet/avazu/day=28/data_0.parquet,118.7,47a5d9133e10369e…
parquet/avazu/day=29/data_0.parquet,85.8,6e813727d953d6da…


Archivo original: train.gz (1.04 GB)
SHA-256 original: 2884596ea9351c8a9abc02bff1f7a4f0d161eac8caceadaaccfff31ec638b4ca
Huella del conjunto de prueba: eb597132a26a3e124529497be401f9ccce223613c644a7a11c707b7889f74a9c


Verificación del conjunto de prueba: correcta


## A.5 Ejecución de extremo a extremo

Desde la raíz del repositorio, con el entorno activado (Windows, PowerShell):

```powershell
python -m src.runner smoke      # prueba de humo con datos sintéticos (unos minutos)
python -m src.runner prepare    # Parquet, manifiesto, huellas y grupos de variables
python -m src.runner sklearn    # Optuna, codificaciones, líneas base, ablaciones y escalado
python -m src.runner spark      # rejilla, modelos, escalado y escalamiento fuerte y débil
python -m src.runner bench      # lectura del CSV y del Parquet con cada motor
python -m src.runner explain    # LIME, SHAP e importancia por permutación
python -m src.runner status     # avance
```

Todas las etapas son reanudables: si una se interrumpe, al volver a ejecutarla omite las corridas terminadas. Después se ejecutan los notebooks de `docs/` en orden (01 a 11) y se compila el libro:

```powershell
jupyter-book build docs/ --all
```

## A.6 Duración de las etapas

In [5]:
log = (C.LOGS / "runner.log").read_text(encoding="utf-8").splitlines() if (C.LOGS / "runner.log").exists() else []
dur = [l for l in log if "terminada en" in l]
tab = pd.DataFrame([{"fecha": l[1:20], "etapa": l.split("Etapa ")[1].split(" terminada")[0], "minutos": float(l.split("en ")[-1].split(" min")[0])} for l in dur])
display(tab.groupby("etapa")["minutos"].sum().round(1).to_frame() if len(tab) else pd.DataFrame({"nota": ["sin registro de duración"]}))
try:
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=C.ROOT).stdout.strip()
    print("Versión del código (commit):", commit or "sin repositorio git")
except Exception:  # noqa: BLE001
    pass

,minutos
etapa,
bench,4.1
explain,1.2
prepare,2.1
sklearn,56.7
spark,414.3


Versión del código (commit): c6961d3
